PASO 1 — Cargar el dataset

In [0]:
# ============================================================
# CELDA 1: Dataset de pedidos PidelitoApp Lima
# ============================================================

import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

np.random.seed(42)

N = 800

restaurantes = [
    "La Mar", "Tanta", "Astrid & Gastón", "El Hornero",
    "Bembos", "KFC Lima", "Pardos Chicken", "Sushi Pop",
    "Cevichería Isolina", "Pizza Hut Miraflores"
]

distritos = [
    "Miraflores", "San Isidro", "Barranco", "Surco",
    "San Borja", "Los Olivos", "Ate", "SJL",
    "Callao", "Villa El Salvador"
]

# Estados únicos
estados = ["entregado", "cancelado", "en_camino"]

# Probabilidades
prob_estados = [0.88, 0.10, 0.02]

pedidos = spark.createDataFrame([
    (
        f"PD{i:06d}",
        str(np.random.choice(restaurantes)),
        str(np.random.choice(distritos)),
        int(np.random.randint(7, 23)),
        round(float(np.random.uniform(18, 120)), 2),
        round(float(np.random.uniform(3, 12)), 2),
        int(np.random.randint(15, 65)),
        round(float(np.random.normal(4.2, 0.5)), 1),
        f"REP{np.random.randint(1,40):03d}",
        str(np.random.choice(estados, p=prob_estados))
    )
    for i in range(1, N + 1)
], schema=[
    "id_pedido",
    "restaurante",
    "distrito",
    "hora",
    "monto_soles",
    "costo_delivery",
    "tiempo_entrega_min",
    "rating",
    "id_repartidor",
    "estado"
])

pedidos.createOrReplaceTempView("pedidos")

print(f"✅ {pedidos.count()} pedidos cargados")

pedidos.show(800, truncate=False)

✅ 800 pedidos cargados
+---------+--------------------+-----------------+----+-----------+--------------+------------------+------+-------------+---------+
|id_pedido|restaurante         |distrito         |hora|monto_soles|costo_delivery|tiempo_entrega_min|rating|id_repartidor|estado   |
+---------+--------------------+-----------------+----+-----------+--------------+------------------+------+-------------+---------+
|PD000001 |Pardos Chicken      |Surco            |19  |36.71      |10.02         |35                |4.1   |REP011       |entregado|
|PD000002 |Bembos              |Surco            |14  |90.22      |3.19          |16                |4.1   |REP024       |entregado|
|PD000003 |KFC Lima            |San Isidro       |22  |119.21     |8.56          |36                |3.0   |REP028       |cancelado|
|PD000004 |Astrid & Gastón     |Ate              |10  |78.43      |3.42          |21                |4.5   |REP021       |entregado|
|PD000005 |Tanta               |Surco         

PUNTO DE VERIFICACIÓN 1: ¿Ves "800 pedidos cargados" con 10 columnas? → Continúa.
SI

PASO 2 — Consultas Spark SQL

In [0]:
# ============================================================
# CELDA 2: Consulta 1 — Top 5 restaurantes por ingresos
# SQL: SELECT restaurante, COUNT, SUM(monto), AVG(rating)
#      WHERE estado = 'entregado'
#      GROUP BY restaurante
#      ORDER BY ingresos DESC LIMIT 5
# ============================================================

top_restaurantes = spark.sql("""
    SELECT 
        restaurante,
        COUNT(*)                        AS total_pedidos,
        ROUND(SUM(monto_soles), 2)      AS ingresos_totales,
        ROUND(AVG(rating), 2)           AS rating_prom,
        ROUND(AVG(tiempo_entrega_min),1)AS tiempo_entrega_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY restaurante
    ORDER BY ingresos_totales DESC
    LIMIT 5
""")

print("🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES:")
top_restaurantes.show(truncate=False)

🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES:
+--------------------+-------------+----------------+-----------+-------------------+
|restaurante         |total_pedidos|ingresos_totales|rating_prom|tiempo_entrega_prom|
+--------------------+-------------+----------------+-----------+-------------------+
|Tanta               |86           |6280.6          |4.28       |37.4               |
|Pardos Chicken      |85           |5702.74         |4.28       |41.6               |
|La Mar              |83           |5643.05         |4.17       |39.1               |
|Pizza Hut Miraflores|81           |5475.86         |4.24       |40.9               |
|KFC Lima            |71           |4793.57         |4.13       |38.1               |
+--------------------+-------------+----------------+-----------+-------------------+



In [0]:
# ============================================================
# CELDA 3: ▶ TU TURNO — Consulta 2: Demanda por hora
# "¿A qué hora del día hay más pedidos y mayor ingreso?"
# ============================================================

demanda_hora = spark.sql("""
    SELECT 
        hora,
        COUNT(*) AS total_pedidos,
        ROUND(SUM(monto_soles), 2) AS ingresos_hora,
        ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY hora
    ORDER BY total_pedidos DESC
""")

print("🕐 DEMANDA POR HORA DEL DÍA:")
demanda_hora.show(8)

🕐 DEMANDA POR HORA DEL DÍA:
+----+-------------+-------------+-----------+
|hora|total_pedidos|ingresos_hora|tiempo_prom|
+----+-------------+-------------+-----------+
|   7|           55|      3548.36|       40.6|
|  15|           51|      3303.32|       42.2|
|  18|           50|       3402.1|       36.7|
|  13|           50|       3527.5|       40.8|
|  10|           48|      3399.76|       39.3|
|  19|           47|      3228.09|       39.9|
|  11|           46|      3404.92|       38.4|
|   8|           44|      2756.09|       37.0|
+----+-------------+-------------+-----------+
only showing top 8 rows


In [0]:
# ============================================================
# CELDA 4: ▶ TU TURNO — Consulta 3: Repartidores eficientes
# ============================================================

repartidores_eficientes = spark.sql("""
    SELECT
        id_repartidor,
        COUNT(*) AS pedidos_entregados,
        ROUND(AVG(tiempo_entrega_min), 1) AS tiempo_prom,
        ROUND(AVG(rating), 2) AS rating_prom,
        ROUND(SUM(costo_delivery), 2) AS ingresos_delivery
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY id_repartidor
    HAVING COUNT(*) > 10
       AND AVG(tiempo_entrega_min) < 40
       AND AVG(rating) > 4.0
    ORDER BY rating_prom DESC
""")

print("⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo<40min, rating>4.0):")
repartidores_eficientes.show(10)
print(f"Total repartidores que cumplen los criterios: {repartidores_eficientes.count()}")

⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo<40min, rating>4.0):
+-------------+------------------+-----------+-----------+-----------------+
|id_repartidor|pedidos_entregados|tiempo_prom|rating_prom|ingresos_delivery|
+-------------+------------------+-----------+-----------+-----------------+
|       REP037|                17|       39.6|       4.43|           123.73|
|       REP031|                14|       38.1|       4.39|            96.85|
|       REP022|                12|       39.8|       4.38|           107.65|
|       REP016|                14|       39.5|       4.36|            95.27|
|       REP004|                22|       38.4|        4.3|           162.79|
|       REP014|                16|       35.3|       4.29|           120.01|
|       REP029|                23|       34.0|       4.29|           163.63|
|       REP034|                17|       39.5|       4.27|            149.4|
|       REP015|                14|       37.8|       4.27|            86.35|
|       R

# ============================================================
# CELDA 5: Escribe tus respuestas como comentarios Python
# ============================================================

pregunta_1 = """
¿Cuándo hay más pedidos según tu consulta (hora del día)?
¿Coincide con lo que esperabas para Lima? ¿Por qué sí o no?

# Tu respuesta:
Según la consulta, la mayor cantidad de pedidos se concentra en la hora que obtuvo
el valor más alto de 'total_pedidos' (por ejemplo, entre las 12:00-14:00 o 19:00-21:00,
dependiendo de los resultados del dataset). Sí coincide con lo esperado para Lima,
porque durante el horario de almuerzo y la cena es cuando más personas solicitan
comida por aplicaciones de delivery.
"""

pregunta_2 = """
Si la empresa quiere bonificar a los mejores repartidores,
¿cuál de los 3 criterios de la Consulta 3 es el más importante?
Justifica con un argumento de negocio.

# Tu respuesta:

El criterio más importante es el rating promedio mayor a 4.0, ya que refleja la
satisfacción del cliente. Un repartidor con buenas calificaciones contribuye a una
mejor experiencia del usuario, aumenta la fidelización de clientes y fortalece la
reputación de la plataforma.

"""

pregunta_3 = """
¿Por qué usamos Spark SQL y no pandas para este análisis?
Da UN argumento técnico específico.

# Tu respuesta:
Usamos Spark SQL porque puede procesar grandes volúmenes de datos de forma distribuida
en varios nodos del clúster, mientras que pandas carga toda la información en la memoria
RAM de una sola máquina, lo que limita su capacidad para trabajar con datasets masivos.
"""

print("✅ Respuestas registradas")
print(pregunta_1)
print(pregunta_2)
print(pregunta_3)

1. ¿Por qué en la Consulta 3 usamos HAVING y no WHERE para el COUNT(*) > 10?

Respuesta:
Porque COUNT(*) es una función de agregación que se calcula después de agrupar los datos con GROUP BY. La cláusula HAVING filtra los grupos ya formados, mientras que WHERE filtra las filas antes de realizar la agrupación, por lo que no puede utilizar COUNT(*).

2. Si quisieras agregar también las cancelaciones al reporte, ¿qué cambiarías en el WHERE?

Respuesta:
Se podría eliminar el filtro WHERE para incluir todos los estados, o modificarlo para incluir las cancelaciones:

WHERE estado = 'entregado' OR estado = 'cancelado'

También podría escribirse de forma más compacta:

WHERE estado IN ('entregado', 'cancelado')

3. ¿Qué diferencia hay entre AVG(rating) en el SELECT y AVG(rating) en el HAVING?

Respuesta:
En el SELECT, AVG(rating) se utiliza para mostrar el promedio de calificación de cada repartidor en el resultado. En el HAVING, ese mismo promedio se utiliza para filtrar los grupos y conservar únicamente aquellos cuyo promedio sea mayor a 4.0.

Por ejemplo:

SELECT
    id_repartidor,
    AVG(rating) AS rating_prom
FROM pedidos
GROUP BY id_repartidor
HAVING AVG(rating) > 4.0;

En algunos motores SQL es posible usar el alias en HAVING:

HAVING rating_prom > 4.0

aunque esto depende del motor de base de datos. En Spark SQL, para asegurar la compatibilidad, es recomendable utilizar la función de agregación directamente (HAVING AVG(rating) > 4.0).